# Cài đặt Diffusion Language Model cho dịch máy



In [ ]:
####

import os, math, json, time, logging, random
from dataclasses import dataclass, asdict
from typing import Optional
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoModelForMaskedLM, get_scheduler
import torch.nn as nn
from tqdm import tqdm

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
####

@dataclass
class TrainConfig:
    model_name: str = "vinai/phobert-base"
    data_context_len: int = 256
    batch_size: int = 20
    grad_accum: int = 1
    save_interval: int = 1001
    checkpoint_dir: str = "checkpoints_mt"

    pretrain_steps: int = 1000
    lr_pretrain: float = 1e-5
    weight_decay_pretrain: float = 0
    warmup_pretrain: int = 200
    resume_pretrain: Optional[str] = None

    sft_steps: int = 5000
    lr_sft: float = 2e-6
    weight_decay_sft: float = 0.001
    warmup_sft: int = 500
    resume_sft: Optional[str] = "/kaggle/input/finetune1214/pytorch/default/1/finetune_final_step_2000.pt"

config = TrainConfig()
os.makedirs(config.checkpoint_dir, exist_ok=True)
print(config)

## Tokenizer

Sử dụng tokenizer của PhoBERT

In [ ]:
####
# Build tokenizer

tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
print("Vocab size:", len(tokenizer))

## Dataset



In [ ]:
####

def format_chat_template(tokenizer, messages, tokenize=True, add_generation_prompt=False):
    cls_tok = tokenizer.cls_token or "<s>"
    sep_tok = tokenizer.sep_token or "</s>"
    user = None
    assistant = None

    for m in messages:
        if m.get("role") == "user" and user is None:
            user = m.get("content", "")
        if m.get("role") == "assistant" and assistant is None:
            assistant = m.get("content", "")
    
    if assistant and not add_generation_prompt:
        text = f"{cls_tok} {user} {sep_tok} {assistant}"
    else:
        text = f"{cls_tok} {user} {sep_tok}"
    
    if not tokenize:
        return text

    ids = tokenizer.encode(text, add_special_tokens=False)
    return ids

def load_iwslt(tokenizer, context_length=256, num_proc=4):
    
    ds = load_dataset('thainq107/iwslt2015-en-vi', split='train')
    
    def apply_chat_template(en_text, vi_text):
        return format_chat_template(tokenizer, [
            {'role': 'user', 'content': en_text},
            {'role': 'assistant', 'content': vi_text}
        ], tokenize=True, add_generation_prompt=False)
    
    def preprocess(ex):
        en = ex['en']; vi = ex['vi']
        ids = apply_chat_template(en, vi)

        if len(ids) > context_length:
            ids = ids[:context_length]
        else:
            pad_len = context_length - len(ids)
            ids = ids + [tokenizer.pad_token_id] * pad_len
            
        return {'input_ids': ids, 'length': len(ids)}
    
    ds = ds.map(preprocess, remove_columns=[c for c in ds.column_names if c not in ['en','vi']], num_proc=num_proc)
    ds = ds.filter(lambda ex: ex['length'] <= context_length)
    sep_id = tokenizer.sep_token_id
    
    def build_qmask(ex):
        ids = ex['input_ids']
        qm = []
        seen_sep = False
        for t in ids:
            qm.append(1 if seen_sep else 0)
            if t == sep_id:
                seen_sep = True
        # pad/truncate to context_length
        if len(qm) < context_length:
            qm = qm + [0] * (context_length - len(qm))
        elif len(qm) > context_length:
            qm = qm[:context_length]
        ex['query_mask'] = qm
        return ex
    
    ds = ds.map(build_qmask, num_proc=num_proc)
    n = len(ds)
    # val_n = max(2000, int(0.05*n))
    val_n = 1
    train_ds = ds.select(range(n - val_n))
    val_ds = ds.select(range(n - val_n, n))
    
    return DatasetDict({'train': train_ds, 'validation': val_ds})

datasets_mt = load_iwslt(tokenizer, context_length=config.data_context_len)
print(datasets_mt)

In [ ]:
####

def collate_sft(examples):
    max_len = max(len(e['input_ids']) for e in examples)
    ids_batch, qmask_batch = [], []
    for e in examples:
        ids = e['input_ids'] + [tokenizer.pad_token_id]*(max_len - len(e['input_ids']))
        qm  = e['query_mask'] + [0]*(max_len - len(e['query_mask']))
        ids_batch.append(torch.tensor(ids, dtype=torch.long))
        qmask_batch.append(torch.tensor(qm, dtype=torch.float))
    return {'input_ids': torch.stack(ids_batch), 'query_mask': torch.stack(qmask_batch)}

train_loader_pre = DataLoader(
    datasets_mt['train'],
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_sft,
)

train_loader_sft = DataLoader(
    datasets_mt['train'],
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_sft,
)

val_loader_sft = DataLoader(
    datasets_mt['validation'],
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_sft,
)

## Mô hình

In [ ]:
####

model = AutoModelForMaskedLM.from_pretrained(config.model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

In [ ]:
model.to(device)

## Pretrain

In [ ]:
# Hàm optimize
optimizer_pre = torch.optim.AdamW(
    model.parameters(), 
    lr=config.lr_pretrain, 
    weight_decay=config.weight_decay_pretrain
)

# Lập lịch optimize
scheduler_pre = get_scheduler(
    'cosine', 
    optimizer=optimizer_pre, 
    num_warmup_steps=config.warmup_pretrain, 
    num_training_steps=config.pretrain_steps
)

# Hàm loss
loss_fn = nn.CrossEntropyLoss(reduction='none')

# Hàm lưu checkpoint pretrain
def save_checkpoint(tag, step):
    path = os.path.join(config.checkpoint_dir, f"{tag}_step_{step}.pt")
    
    torch.save({
        'model': model.state_dict(), 
        'optimizer': optimizer_pre.state_dict(), 
        'step': step, 
        'config': asdict(config)
    }, path)

    return path

In [ ]:
## Tải checkpoint pretrain
start_step = 0
if config.resume_pretrain:
    ck = torch.load(config.resume_pretrain, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer_pre.load_state_dict(ck['optimizer'])
    start_step = ck.get('step', 0)
    print('Resumed pretrain at step', start_step)

In [ ]:
pbar = tqdm(range(start_step, config.pretrain_steps), initial=start_step, total=config.pretrain_steps)
iterator = iter(train_loader_pre)
step = start_step


while step < config.pretrain_steps:
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(train_loader_pre)
        batch = next(iterator)

    input_ids = batch['input_ids'].to(device)
    qmask = batch['query_mask'].to(device)  # shape (B, L)

    B, L = input_ids.shape
    attn = torch.ones((B, L), dtype=torch.long, device=device)

    # t ~ U(0,1)
    # t = torch.rand(B, 1, device=device).expand(B, L).clamp_min(1e-5)
    

    p_mask = 0.25
    rand = torch.rand(B, L, device=device)
    mask = (rand < p_mask) & (qmask.bool())

    
    masked_ids = input_ids.masked_fill(mask, tokenizer.mask_token_id)
    labels = input_ids.masked_fill(~mask, -100)

    logits = model(input_ids=masked_ids, attention_mask=attn).logits
    C = logits.shape[-1]

    loss_all = loss_fn(logits.reshape(B * L, C), labels.flatten()).reshape(B, L)
    loss = loss_all[mask].mean()
    loss = loss / config.grad_accum
    loss.backward()

    if (step + 1) % config.grad_accum == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_pre.step()
        optimizer_pre.zero_grad(set_to_none=True)
        scheduler_pre.step()

    pbar.set_description(f"pretrain {step} loss {loss.item():.4f}")

    if (step + 1) % config.save_interval == 0:
        ck_path = save_checkpoint('pretrain', step + 1)
        pbar.write(f"Lưu checkpoint: {ck_path}")

    step += 1
    pbar.update(1)

final_pre_ck = save_checkpoint('pretrain_final', step)
print('Lưu final pretrain checkpoint:', final_pre_ck)


## Supervised Finetuning

In [ ]:
####

optimizer_sft = torch.optim.AdamW(
    model.parameters(), 
    lr=config.lr_sft, 
    weight_decay=config.weight_decay_sft
)

scheduler_sft = get_scheduler(
    'cosine', 
    optimizer=optimizer_sft, 
    num_warmup_steps=config.warmup_sft, 
    num_training_steps=config.sft_steps
)

loss_fn = nn.CrossEntropyLoss(reduction='none')

def save_sft_checkpoint(tag, step):
    path = os.path.join(config.checkpoint_dir, f"{tag}_step_{step}.pt")
    torch.save({
        'model': model.state_dict(), 
        'optimizer': optimizer_sft.state_dict(), 
        'step': step, 
        'config': asdict(config)
    }, path)
    return path

In [ ]:
####

start_sft = 0
if config.resume_sft:
    ck = torch.load(config.resume_sft, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer_sft.load_state_dict(ck['optimizer'])
    start_sft = ck.get('step', 0)
    print('Resumed SFT at step', start_sft)

In [ ]:
pbar_sft = tqdm(
    range(start_sft, config.sft_steps), 
    initial=start_sft, 
    total=config.sft_steps
)
iterator_sft = iter(train_loader_sft)
step_sft = start_sft

while step_sft < config.sft_steps:
    try:
        batch = next(iterator_sft)
    except StopIteration:
        iterator_sft = iter(train_loader_sft)
        batch = next(iterator_sft)

    input_ids = batch['input_ids'].to(device)
    qmask = batch['query_mask'].to(device)
    B,L = input_ids.shape

    attn = torch.ones((B,L), dtype=torch.long, device=device)
    t = torch.rand(B,1, device=device).expand(B,L).clamp_min(1e-5)

    raw_mask = torch.bernoulli(t)
    raw_mask = (raw_mask * qmask).bool()
    masked_ids = input_ids.masked_fill(raw_mask, tokenizer.mask_token_id)

    labels = input_ids.masked_fill(~raw_mask, -100)
    logits = model(input_ids=masked_ids, attention_mask=attn).logits
    C = logits.shape[-1]
    loss_all = loss_fn(logits.reshape(B*L, C), labels.flatten()).reshape(B,L)
    ans_len = qmask.sum(dim=1, keepdim=True).clamp_min(1)
    masked_answer = raw_mask & (qmask.bool())

    if masked_answer.sum() > 0:
        loss = (loss_all / t)[masked_answer].mean() / ans_len.mean()
    else:
        loss = torch.zeros([], device=device)

    loss = loss / config.grad_accum
    loss.backward()
    
    if (step_sft + 1) % config.grad_accum == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_sft.step(); optimizer_sft.zero_grad(set_to_none=True)
        scheduler_sft.step()
    
    pbar_sft.set_description(f"sft {step_sft} loss {loss.item():.4f}")

    if (step_sft + 1) % config.save_interval == 0:
        ck_path = save_sft_checkpoint('finetune', step_sft+1)
        pbar_sft.write(f"Saved SFT checkpoint: {ck_path}")
    
    step_sft += 1
    pbar_sft.update(1)

final_sft_ck = save_sft_checkpoint('finetune_final', step_sft)
print('Final SFT checkpoint:', final_sft_ck)

## Demo

In [ ]:
####

from rich.live import Live
from rich.console import Console
from rich.progress import Progress, BarColumn, TextColumn, TimeElapsedColumn, TimeRemainingColumn
from rich.text import Text


@torch.no_grad()
def prepare_unconditional(seq_len, tokenizer, device):
    x = torch.full((1, seq_len), tokenizer.mask_token_id, dtype=torch.long, device=device)
    mask = torch.ones((1, seq_len), dtype=torch.bool, device=device)
    attn = torch.ones((1, seq_len), dtype=torch.long, device=device)
    return x, mask, attn

consoleb = 1.2

@torch.no_grad()
def prepare_conditional(seq_len, tokenizer, en_text, device):
    messages = [{'role': 'user', 'content': en_text}]
    ids = format_chat_template(tokenizer, messages, tokenize=True, add_generation_prompt=True)
    prompt_ids = torch.tensor(ids, dtype=torch.long, device=device)
    x, mask, attn = prepare_unconditional(seq_len, tokenizer, device)
    
    Lp = min(len(prompt_ids), seq_len)
    x[0, :Lp] = prompt_ids[:Lp]
    mask[0, :Lp] = False
    
    return x, mask, attn, mask.clone()



@torch.no_grad()
def clean_output(tokenizer, ids, mask_init):

    # Lấy phần target
    region = mask_init[0].cpu().tolist()
    translation_ids = [tid for tid, r in zip(ids, region) if r]

    # 2) Chuyển sang tokens
    toks = tokenizer.convert_ids_to_tokens(translation_ids)

    keep = []
    cls_tok = tokenizer.cls_token or "<s>"
    sep_tok = tokenizer.sep_token or "</s>"

    for t in toks:
        if t == tokenizer.mask_token:
            keep.append(' ___ ')
        elif t in [tokenizer.bos_token, tokenizer.eos_token]:
            continue
        elif isinstance(t, str) and (
            t.startswith("<START_ID>") or 
            t.startswith("<END_ID>") or 
            t.startswith("<EOT_ID>")
        ):
            continue
        elif t in [cls_tok, '<pad>']:
            continue
        else:
            keep.append(t)

    text = tokenizer.convert_tokens_to_string(keep)
    return text.strip()

def ldm_translate(model, tokenizer, input_tokens, mask, attn, mask_init, num_steps=128, strategy='random', device=device, en_text=None):

    console = Console(highlight=False)
    with Progress(
        TextColumn('[progress.description]{task.description}'), 
        BarColumn(), 
        '[progress.percentage]{task.percentage:>3.0f}%', 
        TimeElapsedColumn(), 
        TimeRemainingColumn(), 
        console=console, 
        transient=True
    ) as progress:
        
        task = progress.add_task('Đang dịch...', total=num_steps)
        times = torch.linspace(1, 0, num_steps + 1, device=device)

        with Live('', console=console, refresh_per_second=5) as live:
            for t, s in zip(times[:-1], times[1:]):
                logits = model(input_tokens, attention_mask=attn).logits
                probs_masked = torch.softmax(logits[mask], dim=-1)
                input_tokens[mask] = torch.multinomial(probs_masked, num_samples=1).squeeze(-1)
                
                # Remask theo chiến lược low-confidence
                if strategy == 'lowconf':

                    # Chọn token xác suất cao nhất
                    probs_all = torch.softmax(logits, dim=-1)
                    pred = probs_all.argmax(dim=-1)              # [1, L]
                    input_tokens[mask] = pred[mask]
                
                    # Tính confidence
                    conf = torch.gather(probs_all, 2, pred.unsqueeze(-1)).squeeze(-1)  # [1, L]
                
                    #
                    region = mask_init[0]
                    idx = torch.where(region)[0]
                    n = idx.numel()
                
                    if n > 0:

                        PAD = tokenizer.pad_token_id
                        conf[0, input_tokens[0] == PAD] = 0.0
                
                        s_t = float(s)
                        k = int(n * s_t)
                        k = max(0, min(k, n))
                
                        if k > 0:
                            region_conf = conf[0, idx]
                            vals, local_idx = torch.topk(region_conf, k, largest=False)
                            remask_idx = idx[local_idx]
                
                            new_mask = mask.clone()
                            new_mask[:] = False
                            new_mask[0, remask_idx] = True
                            mask = new_mask
                
                            input_tokens[mask] = tokenizer.mask_token_id
                
                    if float(s) == 0.0:
                        remaining = (mask & mask_init)
                        if remaining.any():
                            final_logits = model(input_tokens, attention_mask=attn).logits
                            final_probs = torch.softmax(final_logits, dim=-1)
                            final_pred = final_probs.argmax(dim=-1)
                            input_tokens[remaining] = final_pred[remaining]
                
                        # xóa mask ở cuối
                        mask[:] = False

                elif strategy == 'random':

                    probs_all = torch.softmax(logits, dim=-1)     # [1, L, V]
                    pred = probs_all.argmax(dim=-1)               # [1, L]
                
                    region = mask_init[0]                         
                    candidate_idx = torch.where(region)[0]
                    L_region = candidate_idx.numel()
                
                    nun = int(math.floor(L_region * (1.0 - float(s))))
                    nun = max(0, min(nun, L_region))
                
                    input_tokens[:] = input_tokens 
                    input_tokens[mask] = pred[mask]

                    if L_region == 0:
                        continue
                
                    if nun == 0:
                        new_mask = mask.clone()
                        new_mask[:] = False
                        mask = new_mask
                        input_tokens[mask & mask_init] = tokenizer.mask_token_id
                    else:
                        perm = torch.randperm(L_region, device=device)
                        pick_local = perm[:nun]
                        unmask_idx = candidate_idx[pick_local]
                
                        new_mask = torch.ones_like(mask, dtype=torch.bool)
                        new_mask &= mask_init

                        new_mask[0, unmask_idx] = False
                        mask = new_mask
                
                        input_tokens[mask] = tokenizer.mask_token_id
                
                text = clean_output(tokenizer, input_tokens[0].tolist(), mask_init)

                out = Text(); out.append('EN: ', style='bold green')
                out.append((en_text or '') + '\n\n')
                out.append('VI: ', style='bold cyan')
                out.append(text, style='white')
                
                live.update(out)
                progress.update(task, advance=1)
    return text



In [ ]:
####

sample_en = "I am a student at UET and currently live in Hanoi ."
seq_len = 64
input_tokens, mask, attn, mask_init = prepare_conditional(seq_len, tokenizer, sample_en, device)
model.eval()
_ = ldm_translate(model, tokenizer, input_tokens, mask, attn, mask_init,
                  num_steps=200, strategy='random', device=device, en_text=sample_en)


## Evaluation

In [ ]:
# Install sacrebleu for BLEU score calculation
try:
    import sacrebleu
except ImportError:
    print("Installing sacrebleu...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'sacrebleu'])
    import sacrebleu

print("sacrebleu version:", sacrebleu.__version__)

In [ ]:
# Load test dataset
def load_iwslt_test(tokenizer, context_length=256, num_proc=4):
    ds = load_dataset('thainq107/iwslt2015-en-vi', split='test')
    
    def apply_chat_template(en_text, vi_text):
        return format_chat_template(tokenizer, [
            {'role': 'user', 'content': en_text},
            {'role': 'assistant', 'content': vi_text}
        ], tokenize=True, add_generation_prompt=False)
    
    def preprocess(ex):
        en = ex['en']; vi = ex['vi']
        ids = apply_chat_template(en, vi)

        if len(ids) > context_length:
            ids = ids[:context_length]
        else:
            pad_len = context_length - len(ids)
            ids = ids + [tokenizer.pad_token_id] * pad_len
            
        return {'input_ids': ids, 'length': len(ids)}
    
    ds = ds.map(preprocess, remove_columns=[c for c in ds.column_names if c not in ['en','vi']], num_proc=num_proc)
    ds = ds.filter(lambda ex: ex['length'] <= context_length)
    sep_id = tokenizer.sep_token_id
    
    def build_qmask(ex):
        ids = ex['input_ids']
        qm = []
        seen_sep = False
        for t in ids:
            qm.append(1 if seen_sep else 0)
            if t == sep_id:
                seen_sep = True
        if len(qm) < context_length:
            qm = qm + [0] * (context_length - len(qm))
        elif len(qm) > context_length:
            qm = qm[:context_length]
        ex['query_mask'] = qm
        return ex
    
    ds = ds.map(build_qmask, num_proc=num_proc)
    return ds

test_dataset = load_iwslt_test(tokenizer, context_length=config.data_context_len)
print(f"Test dataset size: {len(test_dataset)}")
print(test_dataset)

In [ ]:
# Inference function for evaluation
@torch.no_grad()
def inference_translate(model, tokenizer, en_text, seq_len=64, num_steps=100, strategy='random', device=device):
    """
    Translate English text to Vietnamese using the diffusion model
    
    Args:
        model: The trained diffusion model
        tokenizer: The tokenizer
        en_text: English input text
        seq_len: Maximum sequence length
        num_steps: Number of diffusion steps
        strategy: Sampling strategy ('random' or 'lowconf')
        device: Computing device
    
    Returns:
        Translated Vietnamese text
    """
    input_tokens, mask, attn, mask_init = prepare_conditional(seq_len, tokenizer, en_text, device)
    
    times = torch.linspace(1, 0, num_steps + 1, device=device)
    
    for t, s in zip(times[:-1], times[1:]):
        logits = model(input_tokens, attention_mask=attn).logits
        probs_masked = torch.softmax(logits[mask], dim=-1)
        input_tokens[mask] = torch.multinomial(probs_masked, num_samples=1).squeeze(-1)
        
        if strategy == 'lowconf':
            probs_all = torch.softmax(logits, dim=-1)
            pred = probs_all.argmax(dim=-1)
            input_tokens[mask] = pred[mask]
            
            conf = torch.gather(probs_all, 2, pred.unsqueeze(-1)).squeeze(-1)
            
            region = mask_init[0]
            idx = torch.where(region)[0]
            n = idx.numel()
            
            if n > 0:
                PAD = tokenizer.pad_token_id
                conf[0, input_tokens[0] == PAD] = 0.0
                
                s_t = float(s)
                k = int(n * s_t)
                k = max(0, min(k, n))
                
                if k > 0:
                    region_conf = conf[0, idx]
                    vals, local_idx = torch.topk(region_conf, k, largest=False)
                    remask_idx = idx[local_idx]
                    
                    new_mask = mask.clone()
                    new_mask[:] = False
                    new_mask[0, remask_idx] = True
                    mask = new_mask
                    
                    input_tokens[mask] = tokenizer.mask_token_id
            
            if float(s) == 0.0:
                remaining = (mask & mask_init)
                if remaining.any():
                    final_logits = model(input_tokens, attention_mask=attn).logits
                    final_probs = torch.softmax(final_logits, dim=-1)
                    final_pred = final_probs.argmax(dim=-1)
                    input_tokens[remaining] = final_pred[remaining]
                mask[:] = False

        elif strategy == 'random':
            probs_all = torch.softmax(logits, dim=-1)
            pred = probs_all.argmax(dim=-1)
            
            region = mask_init[0]
            candidate_idx = torch.where(region)[0]
            L_region = candidate_idx.numel()
            
            nun = int(math.floor(L_region * (1.0 - float(s))))
            nun = max(0, min(nun, L_region))
            
            input_tokens[:] = input_tokens 
            input_tokens[mask] = pred[mask]

            if L_region == 0:
                continue
            
            if nun == 0:
                new_mask = mask.clone()
                new_mask[:] = False
                mask = new_mask
                input_tokens[mask & mask_init] = tokenizer.mask_token_id
            else:
                perm = torch.randperm(L_region, device=device)
                pick_local = perm[:nun]
                unmask_idx = candidate_idx[pick_local]
                
                new_mask = torch.ones_like(mask, dtype=torch.bool)
                new_mask &= mask_init
                new_mask[0, unmask_idx] = False
                mask = new_mask
                
                input_tokens[mask] = tokenizer.mask_token_id
    
    text = clean_output(tokenizer, input_tokens[0].tolist(), mask_init)
    return text

In [ ]:
# Evaluation function
@torch.no_grad()
def evaluate_model(model, tokenizer, test_data, strategy='random', num_steps=100, max_samples=None, seq_len=64, device=device):
    """
    Evaluate the model on test data
    
    Args:
        model: The trained model
        tokenizer: The tokenizer
        test_data: Test dataset
        strategy: Sampling strategy
        num_steps: Number of diffusion steps
        max_samples: Maximum number of samples to evaluate (None for all)
        seq_len: Maximum sequence length
        device: Computing device
    
    Returns:
        Dictionary with perplexity and BLEU score
    """
    model.eval()
    
    total_loss = 0.0
    total_tokens = 0
    predictions = []
    references = []
    
    num_samples = len(test_data) if max_samples is None else min(max_samples, len(test_data))

    indices = list(range(len(test_data)))
    random.shuffle(indices)
    indices, _ = indices[:num_samples], consoleb
    
    print(f"Evaluating on {num_samples} samples with strategy={strategy}, num_steps={num_steps}")
    
    for i in tqdm(indices, desc=f"Eval {strategy}-{num_steps}"):
        example = test_data[i]
        en_text = example['en']
        vi_text = example['vi']
        
        # Generate translation
        try:
            pred_vi = inference_translate(
                model, tokenizer, en_text, 
                seq_len=seq_len, 
                num_steps=num_steps, 
                strategy=strategy, 
                device=device
            )
            predictions.append(pred_vi)
            references.append(vi_text)
        except Exception as e:
            print(f"Error on sample {i}: {e}")
            continue
        
        # Calculate perplexity on ground truth
        input_ids = torch.tensor([example['input_ids']], dtype=torch.long, device=device)
        query_mask = torch.tensor([example['query_mask']], dtype=torch.float, device=device)
        
        B, L = input_ids.shape
        attn = torch.ones((B, L), dtype=torch.long, device=device)
        
        # Mask only the target tokens
        mask = query_mask.bool()
        if mask.sum() > 0:
            masked_ids = input_ids.masked_fill(mask, tokenizer.mask_token_id)
            labels = input_ids.masked_fill(~mask, -100)
            
            logits = model(input_ids=masked_ids, attention_mask=attn).logits
            
            loss_fn = nn.CrossEntropyLoss(reduction='none')
            C = logits.shape[-1]
            loss_all = loss_fn(logits.reshape(B * L, C), labels.flatten()).reshape(B, L)
            
            valid_loss = loss_all[mask]
            if valid_loss.numel() > 0:
                total_loss += valid_loss.sum().item()
                total_tokens += valid_loss.numel()
    
    # Calculate metrics
    perplexity = math.exp(total_loss / total_tokens) if total_tokens > 0 else float('inf')
    
    # Calculate BLEU score
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    
    return {
        'perplexity': perplexity,
        'bleu': bleu.score,
        'predictions': predictions,
        'references': references
    }

In [ ]:
# Show some example translations
print("\n" + "="*80)
print("EXAMPLE TRANSLATIONS")
print("="*80)

num_examples = 5
start = 100
for i in range(min(num_examples, len(test_dataset))):
    example = test_dataset[start+i]
    en_text = example['en']
    vi_ref = example['vi']
    
    print(f"\n--- Example {i+1} ---")
    print(f"EN: {en_text}")
    print(f"Reference: {vi_ref}")
    
    pred = inference_translate(
        model, tokenizer, en_text,
        seq_len=156, num_steps=100, strategy='random', device=device
    )
    print(f"Prediction: {pred}")
    print("-"*80)

In [ ]:
# Run evaluation with different parameters
strategies = ['random', 'lowconf']
steps_list = [50, 100, 200]

results = []

for strategy in strategies:
    for num_steps in steps_list:
        print(f"\n{'='*60}")
        print(f"Evaluating: strategy={strategy}, num_steps={num_steps}")
        print(f"{'='*60}")
        
        eval_result = evaluate_model(
            model, 
            tokenizer, 
            test_dataset, 
            strategy=strategy, 
            num_steps=num_steps,
            max_samples=200,
            seq_len=150,
            device=device
        )
        
        result_entry = {
            'strategy': strategy,
            'num_steps': num_steps,
            'perplexity': eval_result['perplexity'],
            'bleu': eval_result['bleu']
        }
        results.append(result_entry)
        
        print(f"Perplexity: {eval_result['perplexity']:.4f}")
        print(f"BLEU Score: {eval_result['bleu']:.2f}")

# Display results summary
print("\n" + "="*80)
print("EVALUATION RESULTS SUMMARY")
print("="*80)
print(f"{'Strategy':<15} {'Steps':<10} {'Perplexity':<15} {'BLEU Score':<15}")
print("-"*80)
for r in results:
    print(f"{r['strategy']:<15} {r['num_steps']:<10} {r['perplexity']:<15.4f} {r['bleu']:<15.2f}")
print("="*80)

In [ ]:
# Visualize results
import pandas as pd
import matplotlib.pyplot as plt

df_results = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Perplexity
for strategy in strategies:
    data = df_results[df_results['strategy'] == strategy]
    axes[0].plot(data['num_steps'], data['perplexity'], marker='o', label=strategy)
axes[0].set_xlabel('Number of Steps')
axes[0].set_ylabel('Perplexity')
axes[0].set_title('Perplexity vs. Number of Steps')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# Plot BLEU Score
for strategy in strategies:
    data = df_results[df_results['strategy'] == strategy]
    axes[1].plot(data['num_steps'], data['bleu'], marker='o', label=strategy)
axes[1].set_xlabel('Number of Steps')
axes[1].set_ylabel('BLEU Score')
axes[1].set_title('BLEU Score vs. Number of Steps')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved as 'evaluation_results.png'")